# Notebook 04a: Run Phases 1-3 over the whole dataset and build the feature table (Phase 4, step a)

**Goal:** for every dataset image, run **MediaPipe (Phase 1) -> fretboard detection (Phase 2) -> fingertip-to-string/fret mapping (Phase 3)** and save the result. At the end we have, for each image, a row of **numbers describing where the fingers are on the neck**. Notebook 04b trains a chord classifier on those numbers (and on the images themselves).

**Not in this notebook:** training or any chord prediction. This notebook only *extracts* information.

### Why a separate notebook?
Phase 2 costs about **1.7 seconds per image** on a CPU. The dataset has **3,151 images**, so a full run is roughly **1.5-2.5 hours**. We do it once, save one small JSON per image, and reuse it for every experiment afterwards. That is why this notebook is built to be:
- **Resumable:** every finished image is written to its own file. If Colab disconnects, run the extraction cell again and it skips what is already done.
- **Smoke-testable:** `SMOKE = True` (the default) processes only ~20 images per chord first, so you can see the whole pipeline work in a few minutes before committing to the long run.
- **Parallel:** two worker processes (Colab gives 2 CPU cores) so the long run takes about half the time.

### What the feature table contains
One row per image. Per finger (index, middle, ring, pinky, thumb): the fingertip's **string position** (a number from 1 to 6, continuous, not rounded), its **position along the neck** (in fret-region units), a confidence, and whether it was accepted by Phase 3. Plus **relative** features (differences between fingers), because a chord is a *shape*, not an absolute position. Images where no hand or no neck was found get empty (`NaN`) cells instead of invented numbers; the model in 04b is built to handle missing values.

### Important facts to keep in mind
- MediaPipe finds **no hand in about 43%** of these images, so many rows will have no finger features. That is expected and is reported below.
- Every coordinate is in **original-image pixels**, and the image **ID** is `<Chord>_<split>_<row>` (for example `Am_train_00281`), the same IDs used in notebooks 01-03.

### Cells
0. Setup, 1. Config and data, 2. Copied Phase 1-3 code, 3. Extraction (resumable, parallel), 4. Feature table, 5. Report, 6. Download

## 0. Setup
**What:** creates the folders and installs the two libraries this notebook needs (`mediapipe`, `datasets`). Everything is written under `outputs/phase4/`. No Google Drive.

In [ ]:
# colab-only
import os
PROJECT_DIR = '/content/Guitar-Chord-Recognition'      # temporary folder on the Colab machine
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
print('Project folder:', PROJECT_DIR)

In [ ]:
import os, sys, subprocess, importlib
def _ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', pkg], check=True)
        importlib.invalidate_caches()
_ensure('mediapipe'); _ensure('datasets')
import mediapipe as mp
print('mediapipe', mp.__version__)

## 1. Config and the list of images to process
**What:** one place for the run settings, then the list `ITEMS` of images to process.
- `DATA_SOURCE = 'hf'` (default): download the dataset `j-asra-j/guitar-chords` from Hugging Face inside Colab (no upload needed). `'zip'` uses the small `guitar_chords_sample.zip` (280 images) instead, for quick tests.
- `SMOKE = True`: only `SMOKE_PER_CLASS` images per chord (fixed random seed). **Set `SMOKE = False` for the full 3,151-image run.**
- `N_WORKERS = 2`: processes running in parallel. `TIME_BUDGET_MIN`: stop after this many minutes (and just run the cell again later to continue), useful if you want to do the long run in several sittings.
- IDs are `<Chord>_<split>_<row:05d>`, where `row` is the image's position in its split. The sample zip uses the same IDs, so results line up with notebooks 01-03.

In [ ]:
import json, glob, random, zipfile, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd

DATA_SOURCE = 'hf'          # 'hf' | 'zip'
SMOKE = True                # True: ~20 images per chord first; False: all 3,151 images
SMOKE_PER_CLASS = 20
N_WORKERS = 2               # parallel processes (Colab has 2 CPU cores); 1 = run in this process
TIME_BUDGET_MIN = None      # e.g. 60 = stop after an hour, re-run the extraction cell to continue
SEED = 7

P4 = Path(PROJECT_DIR) / 'outputs' / 'phase4'
FEAT_DIR = P4 / 'features'
FEAT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = f'{PROJECT_DIR}/data/guitar_chords_sample.zip'

DS = {}                      # split -> HF dataset (only for DATA_SOURCE='hf')
ITEMS = []                   # one dict per image: id, chord, split, row (+ zip_name)

if DATA_SOURCE == 'hf':
    from datasets import load_dataset
    R = 'hf://datasets/j-asra-j/guitar-chords@refs%2Fconvert%2Fparquet/default/'
    _ds = load_dataset('parquet', data_files={'train': [R + 'train/0000.parquet', R + 'train/0001.parquet'],
                                               'validation': R + 'validation/0000.parquet', 'test': R + 'test/0000.parquet'})
    NAMES = _ds['train'].features['label'].names
    for split in ('train', 'validation', 'test'):
        DS[split] = _ds[split]
        for row, lab in enumerate(_ds[split]['label']):
            ITEMS.append(dict(id=f'{NAMES[lab]}_{split}_{row:05d}', chord=NAMES[lab], split=split, row=row))
else:
    if not os.path.exists(ZIP_PATH):
        from google.colab import files
        print('Choose guitar_chords_sample.zip ...')
        up = files.upload()
        Path(ZIP_PATH).write_bytes(up[[n for n in up if n.lower().endswith('.zip')][0]])
    with zipfile.ZipFile(ZIP_PATH) as z:
        for zn in sorted(n for n in z.namelist() if n.lower().endswith('.jpg') and '__MACOSX' not in n):
            chord, stem = zn.split('/')[-2], Path(zn).stem            # 'Am', 'train_00281'
            split, row = stem.rsplit('_', 1)
            ITEMS.append(dict(id=f'{chord}_{stem}', chord=chord, split=split, row=int(row), zip_name=zn))
    NAMES = sorted({it['chord'] for it in ITEMS})

assert len({it['id'] for it in ITEMS}) == len(ITEMS), 'image IDs must be unique'
if SMOKE:                                                         # a few images per chord, same ones every run
    rng = random.Random(SEED)
    keep = []
    for ch in NAMES:
        pool = [it for it in ITEMS if it['chord'] == ch]
        keep += rng.sample(pool, min(SMOKE_PER_CLASS, len(pool)))
    ITEMS = sorted(keep, key=lambda it: it['id'])
done = sum((FEAT_DIR / f"{it['id']}.json").exists() for it in ITEMS)
print(f"{len(ITEMS)} images selected ({'SMOKE' if SMOKE else 'FULL'} run, source={DATA_SOURCE}); already extracted: {done}")
print(pd.Series([it['chord'] for it in ITEMS]).value_counts().sort_index().to_dict())

## 2. Phases 1-3 code (copied unchanged from notebook 03)
**What:** three cells of function code, copied **verbatim by the generator script** from `03_fingertip_fret_mapping.ipynb`, which itself copies the Phase 2 detector from notebook 02. Notebooks 01-03 are not modified.
**Why copy instead of import:** Colab notebooks cannot import each other, and re-writing the code would let the versions drift. The fingerprint in the first comment line changes whenever the underlying code changes (re-run `python scripts/make_features_notebook.py` to refresh).

**What the code does, in order (all explained in notebooks 01-03):**
1. `detect_hands`: MediaPipe finds up to 2 hands and 21 landmarks each (pixels in the *original* image, not rounded).
2. `detect_fretboard`: ridge filter + angle comb search + per-string fitting + Hough frets and nut.
3. `map_image`: for the fretting hand's fingertips, signed distance to the detected string and fret lines, ambiguity margins, 12-tone missed-fret repair, and confidence. Each fingertip ends as `accepted`, `tentative` or `rejected`, and is always marked `proximity_only` (nearness in a 2-D photo is not proof of pressing a string).

In [ ]:
# Phase 1/3 settings and imports: copied from notebooks/03_fingertip_fret_mapping.ipynb (fingerprint 1042900f2f). Do not edit here.
import json, glob, random, zipfile, urllib.request
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MAPCFG = dict(
    string_margin_min=0.30,   # 0 = exactly halfway between two strings, 1 = exactly on one string
    fret_margin_min=0.12,     # fraction of the fret-region width that the tip must stay away from a fret line
    outside_frac=0.6,         # beyond the outer strings by more than this x a string gap = not on the neck
    near_neck_frac=0.5,       # when choosing the fretting hand, tips this close (x string gap) to the neck count as near
    accept_conf=0.35,         # minimum overall confidence for 'accepted'
    geometry_quality={'high': 1.0, 'low': 0.6},   # how much we trust Phase 2's own confidence label
    allow_low_geometry=False, # False: low-confidence Phase 2 geometry gives 'tentative' results only
    fret_ratio=2 ** (-1 / 12),# real fret spacing shrinks by this factor per fret (12-tone equal temperament)
    wide_region_ratio=1.8,    # a fret region this many times wider than its neighbours hides a missed fret line
    narrow_region_ratio=0.45, # ... and this many times narrower suggests a spurious line
)

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
MODEL_PATH = f'{PROJECT_DIR}/hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)       # ~8 MB pretrained model from Google

FINGERTIPS = {4: 'Thumb', 8: 'Index', 12: 'Middle', 16: 'Ring', 20: 'Pinky'}
FINGER_COLORS = {'Thumb': (0, 165, 255), 'Index': (0, 255, 0), 'Middle': (255, 128, 0), 'Ring': (255, 0, 255), 'Pinky': (0, 255, 255)}   # BGR, as in Phase 1
CONNECTIONS = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11),
               (11, 12), (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]

def detect_hands(bgr, image_id):
    # Returns the Phase 1 export for one image. All coordinates are ORIGINAL-image pixels (floats).
    h, w = bgr.shape[:2]
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)               # MediaPipe wants RGB, OpenCV gives BGR
    res = LANDMARKER.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    hands = []
    for i, lms in enumerate(res.hand_landmarks):
        cat = res.handedness[i][0]
        pts = [[round(lm.x * w, 2), round(lm.y * h, 2), round(lm.z, 4)] for lm in lms]   # normalized -> pixels
        tips = {name: dict(id=idx, x_px=pts[idx][0], y_px=pts[idx][1], z=pts[idx][2]) for idx, name in FINGERTIPS.items()}
        hands.append(dict(handedness_as_model_sees_it=cat.category_name, score=round(float(cat.score), 3),
                          landmarks_px=pts, tips=tips))
    return dict(schema_version=1, image=dict(id=image_id, width=w, height=h), hands=hands)


def check_coordinates(image_id, bgr, hands, p2):
    # Requirement 4: all three sources must describe the same pixel grid. Returns a list of problems (empty = fine).
    h, w = bgr.shape[:2]
    problems = []
    if (hands['image']['width'], hands['image']['height']) != (w, h):
        problems.append('Phase 1 export size differs from the image')
    if (p2['image']['width'], p2['image']['height']) != (w, h):
        problems.append(f"Phase 2 size {p2['image']['width']}x{p2['image']['height']} differs from the image {w}x{h}")
    if p2.get('image', {}).get('name') != image_id:
        problems.append('Phase 2 image name differs from the image ID')
    if p2.get('fretboard'):
        poly = np.array(p2['fretboard']['polygon'])
        if poly.min() < -2 * max(w, h) or poly.max() > 3 * max(w, h):
            problems.append('Phase 2 polygon is wildly outside the image (wrong coordinate frame?)')
    return problems

In [ ]:
# Phase 2 detector: copied from notebook 03 / notebook 02 (fingerprint 1042900f2f). Do not edit here.
# Phase 2 detector: copied from notebooks/02_fretboard_detection.ipynb (fingerprint 3bc032056c). Do not edit here.
import os, json, glob, random
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CFG = dict(
    max_side=800,            # resize so the longest image side is at most this many pixels (speed + consistent scale)
    canny_sigma=0.33,        # Canny thresholds are set from the image's median brightness (used for the fret lines)
    hough_threshold=40,      # votes a line needs in the Hough transform (higher = fewer, surer lines)
    hough_min_len_frac=0.08, # shortest accepted segment, as a fraction of the longest image side
    hough_max_gap=8,         # gaps (px) in an edge line that are bridged
    ridge_sigmas=(1.2, 2.0), # line widths (px) the ridge filter looks for; thin strings, slightly thicker fret wires
    angle_step=2.0,         # the neck-direction search tries every this-many degrees
    n_angle_candidates=4,    # how many different neck directions are fully tested; the best-scoring one wins
    string_gap_min_frac=0.015,  # smallest allowed spacing between neighbouring strings (x longest image side)
    string_gap_max_frac=0.09,   # largest allowed spacing between neighbouring strings
    string_min_extent_frac=0.2, # strings must run along at least this fraction of the image (rejects small noisy patches)
    string_min_density=0.5,     # at least this fraction of each string's length must show real line evidence
    fret_max_tilt=30,        # frets must be within this many degrees of perpendicular to the strings
    fret_min_cover=0.5,      # a fret must span at least this fraction of the neck width
)

# Where outputs go. PROJECT_DIR is defined in cell 0.


def show_images(images, titles, cols=3, size=4):
    # Display a list of images in a grid. Grayscale images (2D) are shown in gray.
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size))
    axes = np.atleast_1d(axes).ravel()
    for ax, im, t in zip(axes, images, titles):
        if im.ndim == 2:
            ax.imshow(im, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))   # OpenCV is BGR, matplotlib is RGB
        ax.set_title(t, fontsize=9); ax.axis('off')
    for ax in axes[len(images):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()


def ridge_response(gray, valid):
    # Highlights thin bright or dark lines (strings, frets) as one response each, scaled to 0..1.
    g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray).astype(np.float32) / 255.0   # local contrast boost
    best = np.zeros_like(g)
    for s in CFG['ridge_sigmas']:                                    # two line widths; keep the stronger response
        gs = cv2.GaussianBlur(g, (0, 0), s)
        dxx = cv2.Sobel(gs, cv2.CV_32F, 2, 0, ksize=3)               # second derivatives = how sharply brightness bends
        dyy = cv2.Sobel(gs, cv2.CV_32F, 0, 2, ksize=3)
        dxy = cv2.Sobel(gs, cv2.CV_32F, 1, 1, ksize=3)
        t = np.sqrt(((dxx - dyy) / 2) ** 2 + dxy ** 2)
        l1 = (dxx + dyy) / 2 + t                                     # the two eigenvalues of the Hessian (l1 >= l2)
        l2 = (dxx + dyy) / 2 - t
        bright = np.maximum(np.where(np.abs(l1) < 0.5 * np.abs(l2), -l2, 0), 0)   # thin BRIGHT line: one strongly negative curvature
        dark = np.maximum(np.where(np.abs(l2) < 0.5 * np.abs(l1), l1, 0), 0)      # thin DARK line: one strongly positive curvature
        best = np.maximum(best, (bright + dark) * s ** 2)            # s**2 makes the two scales comparable
    best *= (valid > 0)
    ref = np.percentile(best[valid > 0], 97) if (valid > 0).any() else 1.0
    return np.clip(best / max(ref, 1e-6), 0, 1).astype(np.float32)   # clipping stops one huge edge from dominating


def preprocess(bgr, max_side=800):
    h0, w0 = bgr.shape[:2]
    scale = min(1.0, max_side / max(h0, w0))                       # never enlarge, only shrink
    img = cv2.resize(bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else bgr.copy()

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)                        # 5x5 kernel, sigma chosen automatically

    med = float(np.median(blur))
    lo = int(max(20, (1 - CFG['canny_sigma']) * med))
    hi = int(max(60, (1 + CFG['canny_sigma']) * med))
    edges_raw = cv2.Canny(blur, lo, hi)

    # Mask out the black padding that rotation-augmented images have, and the edge where it meets the photo
    valid = (gray > 8).astype(np.uint8)
    valid = cv2.morphologyEx(valid, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    valid = cv2.erode(valid, np.ones((13, 13), np.uint8))
    edges = cv2.bitwise_and(edges_raw, edges_raw, mask=valid)

    ridge = ridge_response(gray, valid)
    return dict(img=img, gray=gray, blur=blur, edges_raw=edges_raw, edges=edges, ridge=ridge, scale=scale, canny=(lo, hi))


def seg_lengths(segs):
    return np.hypot(segs[:, 2] - segs[:, 0], segs[:, 3] - segs[:, 1])

def seg_angles(segs):
    # direction of each segment in degrees, in [0, 180). 0 = pointing right, 90 = pointing down.
    return np.degrees(np.arctan2(segs[:, 3] - segs[:, 1], segs[:, 2] - segs[:, 0])) % 180.0

def signed_angle(deg):
    # wraps angles to [-90, 90) so "almost horizontal" is near 0 whichever way the line points
    return (deg + 90.0) % 180.0 - 90.0


def hough_segments(edges):
    h, w = edges.shape
    lines = cv2.HoughLinesP(edges, rho=1, theta=np.pi / 180, threshold=CFG['hough_threshold'],
                            minLineLength=int(CFG['hough_min_len_frac'] * max(h, w)),
                            maxLineGap=CFG['hough_max_gap'])
    if lines is None:
        return np.zeros((0, 4))
    return lines.reshape(-1, 4).astype(float)     # shape differs between OpenCV versions; this handles both


def neck_matrix(angle, w, h):
    # Rotation that makes lines pointing at `angle` degrees horizontal. The output canvas is the image diagonal wide,
    # so nothing is cropped when the photo is rotated. Returns the 2x3 matrix and the canvas size.
    D = int(np.ceil(np.hypot(w, h)))
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    M[0, 2] += D / 2 - w / 2
    M[1, 2] += D / 2 - h / 2
    return M, D


def to_neck_frame(segs, M):
    # Apply the same rotation to segment endpoints.
    pts = segs.reshape(-1, 2) @ M[:, :2].T + M[:, 2]
    return pts.reshape(-1, 4)


def comb_search(profile, d_min, d_max, n=6, tol=2):
    # profile: ridge evidence per row. Find the best place and spacing d for n equally spaced teeth.
    p = np.convolve(profile, np.ones(3) / 3, mode='same')
    base = np.convolve(p, np.ones(41) / 41, mode='same')                  # slowly varying background level
    p = np.maximum(p - base, 0).astype(np.float32)                         # keep only peaks standing above it
    pd_ = cv2.dilate(p.reshape(-1, 1), np.ones((2 * tol + 1, 1), np.uint8)).ravel()   # each tooth may be +-tol rows off
    best = (0.0, 0, 0)                                                     # (score, first row, spacing)
    for d in range(d_min, d_max + 1):
        offs = np.arange(n) * d
        starts = np.arange(0, len(pd_) - offs[-1])
        if len(starts) == 0:
            break
        v = np.sort(pd_[starts[:, None] + offs[None, :]], axis=1)          # evidence under each tooth, weakest first
        # drop the weakest tooth (a string may be hidden) but reward teeth of similar strength
        score = v[:, 1:].sum(1) * np.sqrt(v[:, 1] / (v[:, -1] + 1e-9))
        i = int(np.argmax(score))
        if score[i] > best[0]:
            best = (float(score[i]), int(starts[i]), d)
    return best


def neck_candidates(ridge, n=None):
    # Returns the best few distinct neck directions as dicts, and (angles, scores) for plotting.
    n = n or CFG['n_angle_candidates']
    h, w = ridge.shape
    big = max(h, w)
    d_min, d_max = max(8, int(CFG['string_gap_min_frac'] * big)), int(CFG['string_gap_max_frac'] * big)
    phis = np.arange(0, 180, CFG['angle_step'])
    scores, info = np.zeros(len(phis)), []
    for i, phi in enumerate(phis):
        M, D = neck_matrix(phi, w, h)
        rot = cv2.warpAffine(ridge, M, (D, D))                             # ridge map with direction phi made horizontal
        coherent = np.maximum(cv2.blur(rot, (41, 1)) - 0.35, 0)            # keep only long unbroken lines
        sc, y_start, d = comb_search(coherent.sum(axis=1), d_min, d_max)
        scores[i] = sc
        info.append((y_start, d))
    picked = []
    for i in np.argsort(scores)[::-1]:                                     # best first; skip directions too close to a pick
        if scores[i] <= 0:
            break
        if all(min(abs(phis[i] - phis[j]), 180 - abs(phis[i] - phis[j])) >= 10 for j in picked):
            picked.append(int(i))
        if len(picked) == n:
            break
    cands = [dict(angle=float(phis[i]), comb_score=float(scores[i]), y_start=info[i][0], d=info[i][1]) for i in picked]
    return cands, (phis, scores)


def string_y(st, x):
    # y position of a string at horizontal position x (strings are lines y = y0 + m*(x - xc))
    return st['y0'] + st['m'] * (x - st['xc'])


def _runs(mask):
    # (start, end) of each run of True values in a 1-D boolean array
    edges = np.diff(np.r_[0, mask.astype(np.int8), 0])
    return list(zip(np.nonzero(edges == 1)[0], np.nonzero(edges == -1)[0] - 1))


def find_strings(ridge, cand, n_strings=6):
    # Refine the comb's rough guess into 6 fitted string lines in the neck-aligned view. Returns a dict, or None.
    h, w = ridge.shape
    M, D = neck_matrix(cand['angle'], w, h)
    rot = cv2.warpAffine(ridge, M, (D, D))
    ys, xs = np.nonzero(rot > 0.25)                                        # ridge pixels worth fitting to
    wt = rot[ys, xs].astype(float); ys = ys.astype(float); xs = xs.astype(float)
    d = cand['d']
    y_init = [cand['y_start'] + i * d for i in range(n_strings)]
    near = np.zeros(len(ys), bool)
    for yi in y_init:
        near |= np.abs(ys - yi) <= 0.35 * d
    if near.sum() < 50:
        return None
    xc = float(np.average(xs[near], weights=wt[near]))                     # shared pivot column in the middle of the evidence

    # 1. per-string slope search (perspective makes strings fan out)
    slopes = []
    for yi in y_init:
        scores = [(wt[np.abs(ys - (yi + t * (xs - xc))) <= 2.0].sum(), t) for t in np.tan(np.radians(np.arange(-12, 12.1, 1.0)))]
        slopes.append(max(scores)[1])
    # 2. slopes should vary smoothly across the 6 strings: fit a line through them and stay close to it
    idx = np.arange(n_strings)
    a, b = np.polyfit(idx, slopes, 1)
    model = a * idx + b
    lines = []
    for i, yi in enumerate(y_init):
        y0, m = yi, float(model[i])
        # 3. weighted least squares on nearby ridge pixels, with a narrowing band
        for hw in (0.30 * d, 0.22 * d, 0.16 * d):
            sel = np.abs(ys - (y0 + m * (xs - xc))) <= hw
            if sel.sum() < 25:
                break
            A = np.polyfit(xs[sel] - xc, ys[sel], 1, w=wt[sel])
            m = float(np.clip(A[0], model[i] - 0.04, model[i] + 0.04))     # at most ~2.3 degrees away from the fan model
            y0 = float(A[1])
        lines.append((y0, m))

    # 4. where along the neck is there real evidence? (a string may be hidden by fingers over part of its length)
    X = np.arange(D)
    support, count = [], np.zeros(D)
    for y0, m in lines:
        yy = np.rint(y0 + m * (X - xc)).astype(int)
        ok = (yy >= 1) & (yy < D - 1)
        v = np.zeros(D)
        for dy in (-1, 0, 1):
            v[ok] = np.maximum(v[ok], rot[yy[ok] + dy, X[ok]])
        s = cv2.morphologyEx((v > 0.2).astype(np.uint8).reshape(1, -1), cv2.MORPH_CLOSE, np.ones((1, 31), np.uint8)).ravel()
        support.append(s); count += s
    good = cv2.morphologyEx((count >= 4).astype(np.uint8).reshape(1, -1), cv2.MORPH_CLOSE, np.ones((1, 41), np.uint8)).ravel()
    runs = _runs(good > 0)
    if not runs:
        return None
    xl, xr = max(runs, key=lambda r: count[r[0]:r[1] + 1].sum())
    strings = []
    for (y0, m), s in zip(lines, support):
        dens = float(s[xl:xr + 1].mean())
        strings.append(dict(xc=xc, y0=y0, m=m, x1=float(xl), x2=float(xr), length=float((xr - xl) * dens), density=dens))

    # 5. sanity checks: strings must stay in order (no crossing) over the whole stretch, with fairly even spacing
    gaps_l = np.diff([string_y(s, xl) for s in strings]); gaps_r = np.diff([string_y(s, xr) for s in strings])
    if gaps_l.min() < 2 or gaps_r.min() < 2:
        return None
    gaps = np.diff([s['y0'] for s in strings])
    cv = float(gaps.std() / gaps.mean())
    return dict(M=M, strings=strings, cv=cv, xl=float(xl), xr=float(xr), extent=float(xr - xl),
                density=float(np.mean([s['density'] for s in strings])))


def merge_intervals(iv):
    # total length covered by a set of (start, end) intervals, counting overlaps once
    iv = sorted(iv); total, cur_s, cur_e = 0.0, None, None
    for a, b in iv:
        if cur_e is None or a > cur_e:
            if cur_e is not None:
                total += cur_e - cur_s
            cur_s, cur_e = a, b
        else:
            cur_e = max(cur_e, b)
    if cur_e is not None:
        total += cur_e - cur_s
    return total


def find_frets(sr, strings, spacing):
    # strings: the 6 chosen strings. spacing: mean distance between neighboring strings.
    xc = strings[0]['xc']
    y_top = strings[0]['y0'] - spacing / 2          # neck band edges at x = xc
    y_bot = strings[-1]['y0'] + spacing / 2
    band = y_bot - y_top
    y_mid = (y_top + y_bot) / 2

    ang = np.abs(signed_angle(seg_angles(sr)))
    s = sr[ang >= 90 - CFG['fret_max_tilt']].copy()                       # keep almost-vertical segments
    if len(s) == 0:
        return [], dict(y_top=y_top, y_bot=y_bot, y_mid=y_mid)
    swap = s[:, 1] > s[:, 3]                                              # make every segment point downward
    s[swap] = s[swap][:, [2, 3, 0, 1]]

    items = []
    for x1, y1, x2, y2 in s:
        if y2 - y1 < 1e-6:
            continue
        a, b = max(y1, y_top), min(y2, y_bot)                              # part of the segment inside the neck band
        if b - a < 0.25 * band:
            continue
        k = (x2 - x1) / (y2 - y1)                                          # horizontal shift per unit of y
        items.append((x1 + k * (y_mid - y1), k, a, b, y2 - y1))            # (x at band middle, k, y-range, full length)
    if not items:
        return [], dict(y_top=y_top, y_bot=y_bot, y_mid=y_mid)

    items.sort(key=lambda t: t[0])
    tol = max(3.0, 0.8 * spacing)
    groups, cur = [], [items[0]]
    for it in items[1:]:
        if it[0] - cur[-1][0] <= tol:
            cur.append(it)
        else:
            groups.append(cur); cur = [it]
    groups.append(cur)

    frets = []
    for g in groups:
        cover = merge_intervals([(a, b) for _, _, a, b, _ in g]) / band
        if cover < CFG['fret_min_cover']:
            continue
        wgt = np.array([b - a for _, _, a, b, _ in g])
        frets.append(dict(xm=float(np.average([t[0] for t in g], weights=wgt)),
                          k=float(np.average([t[1] for t in g], weights=wgt)), cover=float(cover),
                          span=float(max(t[4] for t in g))))             # longest piece seen = real neck width
    return frets, dict(y_top=y_top, y_bot=y_bot, y_mid=y_mid)


def pick_nut(frets, strings):
    # Returns (nut or None, remaining frets). Needs 3+ lines so we can compare the gaps at both ends.
    if len(frets) < 3:
        return None, frets
    xs = [f['xm'] for f in frets]
    first_gap, last_gap = xs[1] - xs[0], xs[-1] - xs[-2]
    if first_gap >= last_gap:                                  # wider gaps on the left -> headstock on the left
        nut, rest, gap = frets[0], frets[1:], first_gap
        continues = min(st['x1'] for st in strings) <= nut['xm'] - 0.5 * gap
    else:
        nut, rest, gap = frets[-1], frets[:-1], last_gap
        continues = max(st['x2'] for st in strings) >= nut['xm'] + 0.5 * gap
    nut = dict(nut, confidence='medium' if continues else 'low')
    return nut, rest


def _to_original(pts_rot, M, scale):
    inv = cv2.invertAffineTransform(M)
    p = np.asarray(pts_rot, float).reshape(-1, 2) @ inv[:, :2].T + inv[:, 2]
    return (p / scale).round(1).tolist()


def _try_angle(pre, segs, cand):
    # Run strings -> frets -> nut for ONE candidate neck direction and give the result a score for ranking.
    h, w = pre['img'].shape[:2]
    big = max(h, w)
    out = dict(angle=cand['angle'], M=None, strings=[], cv=None, n_frets=0, score=0.0, density=0.0, comb_score=cand['comb_score'])
    fit = find_strings(pre['ridge'], cand)
    if fit is None:
        out.update(status='failed', reason='no 6 evenly spaced, non-crossing strings found along this direction')
        return out
    out.update(M=fit['M'], strings=fit['strings'], cv=fit['cv'], density=fit['density'], xl=fit['xl'], xr=fit['xr'])
    if fit['extent'] < CFG['string_min_extent_frac'] * big or fit['density'] < CFG['string_min_density']:
        out.update(status='failed', reason=f"strings too short ({fit['extent'] / big:.0%} of the image) or too patchy (density {fit['density']:.2f})")
        return out

    strings = fit['strings']
    spacing = float(np.mean(np.diff([c['y0'] for c in strings])))
    sr = to_neck_frame(segs, fit['M'])
    frets, band = find_frets(sr, strings, spacing)
    nut, rest = pick_nut(frets, strings)
    n_lines = len(rest) + (1 if nut else 0)
    # Cross-check: the strings should span about as much width as the fret lines do. If the strings cover much less,
    # we probably grabbed the wrong lines (for example a patch of wood grain).
    all_lines = rest + ([nut] if nut else [])
    ratio = (band['y_bot'] - band['y_top']) / float(np.median([f['span'] for f in all_lines])) if all_lines else None
    # Ranking score: the comb strength, boosted when fret lines cross the strings (a real neck has them, wood grain does not)
    # and by how continuous the string evidence is.
    score = cand['comb_score'] * (1 + 0.3 * min(n_lines, 6)) * fit['density']
    out.update(spacing=spacing, band=band, frets=rest, nut=nut, n_frets=n_lines, band_ratio=ratio, score=score)

    if n_lines >= 2:
        out.update(status='ok', reason=None)
    else:
        out.update(status='partial', reason=f'6 strings found but only {n_lines} fret line(s)')
    return out


def detect_fretboard(bgr, name='image'):
    pre = preprocess(bgr, CFG['max_side'])
    segs = hough_segments(pre['edges'])
    H0, W0 = bgr.shape[:2]
    res = dict(schema_version=1, image=dict(name=name, width=W0, height=H0), status='failed', success=False,
               confidence='none', failure_reason=None, neck_angle_deg=None,
               fretboard=None, strings=[], frets=[], nut=None, diagnostics=dict(n_segments=int(len(segs))))
    cands, _ = neck_candidates(pre['ridge'])
    if not cands:
        res['failure_reason'] = 'no line pattern found (the ridge map is empty)'
        return res

    tries = [_try_angle(pre, segs, c) for c in cands]
    best = max(tries, key=lambda t: t['score'])
    if best['score'] <= 0:                              # nothing valid at all: report the most informative failure
        best = tries[0]

    res['status'], res['success'] = best['status'], best['status'] == 'ok'
    res['failure_reason'], res['neck_angle_deg'] = best['reason'], round(float(best['angle']), 1)
    ratio = best.get('band_ratio')
    res['diagnostics'].update(n_angle_candidates=len(cands), comb_score=round(best['comb_score'], 1),
                              string_density=round(best['density'], 2),
                              string_band_vs_fret_ratio=None if ratio is None else round(float(ratio), 2),
                              string_spacing_cv=None if best['cv'] is None else round(best['cv'], 3),
                              n_fret_lines=best['n_frets'])
    if best['status'] == 'failed':
        return res

    M, scale, strings, sp = best['M'], pre['scale'], best['strings'], best['spacing']
    # Neck extent along the strings comes from where the strings have evidence. Fret lines can extend it a little
    # (frets are sometimes visible where the strings are hidden), but a stray fret far away is ignored.
    xl, xr, basis = best['xl'], best['xr'], 'strings'
    margin = 0.15 * (xr - xl)
    near_x = [f['xm'] for f in best['frets'] + ([best['nut']] if best['nut'] else []) if xl - margin <= f['xm'] <= xr + margin]
    if near_x and (min(near_x) < xl or max(near_x) > xr):
        xl, xr, basis = min(xl, min(near_x)), max(xr, max(near_x)), 'strings+frets'

    top_y = lambda x: string_y(strings[0], x) - sp / 2
    bot_y = lambda x: string_y(strings[-1], x) + sp / 2
    band = best.get('band')

    def fret_ends(f):
        yt, yb = top_y(f['xm']), bot_y(f['xm'])
        return [(f['xm'] + f['k'] * (yt - band['y_mid']), yt), (f['xm'] + f['k'] * (yb - band['y_mid']), yb)]

    res['strings'] = [dict(index=i + 1, p1=_to_original([(xl, string_y(st, xl))], M, scale)[0],
                           p2=_to_original([(xr, string_y(st, xr))], M, scale)[0]) for i, st in enumerate(strings)]
    kept = [f for f in best['frets'] if xl - 1 <= f['xm'] <= xr + 1]
    res['frets'] = [dict(index=i + 1, p1=_to_original([fret_ends(f)[0]], M, scale)[0],
                         p2=_to_original([fret_ends(f)[1]], M, scale)[0]) for i, f in enumerate(kept)]
    if best['nut']:
        e = fret_ends(best['nut'])
        res['nut'] = dict(p1=_to_original([e[0]], M, scale)[0], p2=_to_original([e[1]], M, scale)[0],
                          confidence=best['nut']['confidence'])
    corners = [(xl, top_y(xl)), (xr, top_y(xr)), (xr, bot_y(xr)), (xl, bot_y(xl))]
    res['fretboard'] = dict(polygon=_to_original(corners, M, scale), basis=basis)

    # Confidence: 'ok' only means "geometry was produced". A result is trusted ('high') only if several fret lines
    # were found, the neck spans a large part of the image, the strings cover a sensible share of the neck width,
    # AND the strings show continuous line evidence. Small patches of background texture fail this.
    poly = np.array(res['fretboard']['polygon'])
    length_frac = float(np.hypot(*(poly[1] - poly[0]))) / max(W0, H0)
    res['diagnostics']['neck_length_frac'] = round(length_frac, 2)
    ratio_ok = ratio is not None and ratio >= 0.6              # strings must cover a sensible share of the neck width
    if best['status'] == 'ok' and best['n_frets'] >= 4 and length_frac >= 0.4 and ratio_ok and best['density'] >= 0.6:
        res['confidence'] = 'high'
    else:
        res['confidence'] = 'low'
    return res


COL = dict(region=(0, 165, 255), string=(255, 255, 0), fret=(0, 255, 0), nut=(255, 0, 255))   # BGR colors


def draw_result(bgr, res, alpha=0.22):
    # Draw a result dictionary (works on one loaded from JSON too) on top of the original image.
    out = bgr.copy()
    t = max(1, int(round(max(out.shape[:2]) / 500)))
    P = lambda p: (int(round(p[0])), int(round(p[1])))
    if res.get('fretboard'):
        poly = np.array(res['fretboard']['polygon'], np.int32)
        ov = out.copy(); cv2.fillPoly(ov, [poly], COL['region'])
        out = cv2.addWeighted(ov, alpha, out, 1 - alpha, 0)
        cv2.polylines(out, [poly], True, COL['region'], t + 1, cv2.LINE_AA)
    for s in res.get('strings', []):
        cv2.line(out, P(s['p1']), P(s['p2']), COL['string'], t, cv2.LINE_AA)
    for f in res.get('frets', []):
        cv2.line(out, P(f['p1']), P(f['p2']), COL['fret'], t + 1, cv2.LINE_AA)
    if res.get('nut'):
        cv2.line(out, P(res['nut']['p1']), P(res['nut']['p2']), COL['nut'], t + 2, cv2.LINE_AA)
    return out

In [ ]:
# Phase 3 geometry and mapping: copied from notebook 03 (fingerprint 1042900f2f). Do not edit here.
def _unit(v):
    v = np.asarray(v, float)
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def _mid(line):
    return (line['p1'] + line['p2']) / 2


def signed_dist(line, P, ref_dir):
    # Perpendicular distance from P to the (infinite) line through p1,p2, positive on the side ref_dir points to.
    d = line['p2'] - line['p1']
    n = _unit([-d[1], d[0]])                      # a normal vector (rotate the direction by 90 degrees)
    if n @ ref_dir < 0:
        n = -n
    return float((P - line['p1']) @ n)


def build_neck_model(p2):
    # Turn a Phase 2 result into the objects the mapping needs. Returns None if there are no usable strings.
    if p2.get('status') == 'failed' or len(p2.get('strings', [])) != 6 or not p2.get('fretboard'):
        return None
    S = [dict(p1=np.array(s['p1'], float), p2=np.array(s['p2'], float)) for s in sorted(p2['strings'], key=lambda s: s['index'])]
    cen = [_mid(s) for s in S]
    c_dir = _unit(cen[-1] - cen[0])                                  # across the neck: string 1 -> string 6
    spacing = float(np.mean(np.diff([(c - cen[0]) @ c_dir for c in cen])))
    if spacing <= 1:
        return None
    a_rough = _unit(np.mean([s['p2'] - s['p1'] for s in S], axis=0))  # along the neck (left end -> right end of Phase 2's view)

    lines = [dict(p1=np.array(f['p1'], float), p2=np.array(f['p2'], float), kind='fret', virtual=False) for f in p2.get('frets', [])]
    nut_conf = None
    if p2.get('nut'):
        lines.append(dict(p1=np.array(p2['nut']['p1'], float), p2=np.array(p2['nut']['p2'], float), kind='nut', virtual=False,
                          conf=p2['nut'].get('confidence')))
    model = dict(strings=S, c_dir=c_dir, spacing=spacing, polygon=np.array(p2['fretboard']['polygon'], float),
                 boundaries=[], b_dir=None, orientation='none', nut_conf=None, n_virtual=0)
    if len(lines) < 2:
        return model                                                  # strings only: we can name the string but not the fret

    lines.sort(key=lambda l: _mid(l) @ a_rough)                       # left to right along the neck
    # Decide which end is the headstock end. Prefer the nut; otherwise the end with wider gaps.
    nut_idx = [i for i, l in enumerate(lines) if l['kind'] == 'nut']
    pos = [_mid(l) @ a_rough for l in lines]
    gaps = np.diff(pos)
    orientation = 'unknown'
    if nut_idx and nut_idx[0] in (0, len(lines) - 1):
        if nut_idx[0] == len(lines) - 1:
            lines = lines[::-1]
        orientation = 'nut'
        model['nut_conf'] = lines[0].get('conf')
    elif len(lines) >= 3:
        if gaps[0] < gaps[-1]:
            lines = lines[::-1]
        orientation = 'gap_trend'
    # Repair missed frets with the 12-tone spacing rule (gaps should shrink by ~0.944 per fret).
    ordered, unit_gap = [lines[0]], None
    r12 = MAPCFG['fret_ratio']
    for j in range(1, len(lines)):
        g = float(np.linalg.norm(_mid(lines[j]) - _mid(lines[j - 1])))
        if j == 1 and orientation != 'unknown' and len(lines) >= 3:
            # The FIRST gap has no earlier gap to compare with, so compare it with the NEXT one. If a gap holds mm+1 real
            # frets, its width relative to the next region is (1 + r + ... + r^mm) / r^(mm+1): 1.06, 2.2, 3.4, ...
            g2 = float(np.linalg.norm(_mid(lines[2]) - _mid(lines[1])))
            R = g / max(g2, 1e-6)
            expect = [sum(r12 ** i for i in range(mm + 1)) / r12 ** (mm + 1) for mm in range(5)]
            mm = int(np.argmin([abs(np.log(max(R, 1e-6) / e)) for e in expect]))
            if mm >= 1 and R > 1.5:
                tot = sum(r12 ** i for i in range(mm + 1))
                for t in np.cumsum([r12 ** i for i in range(mm)]) / tot:     # missed frets sit at the 12-tone proportions
                    ordered.append(dict(p1=lines[0]['p1'] * (1 - t) + lines[1]['p1'] * t,
                                        p2=lines[0]['p2'] * (1 - t) + lines[1]['p2'] * t, kind='fret', virtual=True))
                    model['n_virtual'] += 1
                ordered.append(lines[1])
                unit_gap = g * r12 ** mm / tot
                continue
        m = 1
        if unit_gap is not None and orientation != 'unknown':
            expected = unit_gap * MAPCFG['fret_ratio']
            m = int(np.clip(round(g / expected), 1, 3))
            if m < 2 or g < 1.5 * expected:
                m = 1
        for i in range(1, m):                                         # virtual boundaries, evenly spread inside the gap
            t = i / m
            ordered.append(dict(p1=lines[j - 1]['p1'] * (1 - t) + lines[j]['p1'] * t,
                                p2=lines[j - 1]['p2'] * (1 - t) + lines[j]['p2'] * t, kind='fret', virtual=True))
            model['n_virtual'] += 1
        ordered.append(lines[j])
        unit_gap = g / m
    model.update(boundaries=ordered, orientation=orientation,
                 b_dir=_unit(_mid(ordered[-1]) - _mid(ordered[0])))
    # Irregular spacing check: a real fret region is only ~6% narrower than the one before it. A region much wider than
    # its neighbours hides a missed fret (or two); a much narrower one means a spurious line. We cannot name the fret there.
    widths = [float(np.linalg.norm(_mid(ordered[k]) - _mid(ordered[k - 1]))) for k in range(1, len(ordered))]   # slot k = widths[k-1]
    suspect = {}
    for k in range(1, len(ordered)):
        if ordered[k]['virtual'] or ordered[k - 1]['virtual']:
            continue
        nb = [widths[j - 1] for j in (k - 1, k + 1) if 1 <= j <= len(widths)]
        if not nb:
            continue
        ratio = widths[k - 1] / max(float(np.mean(nb)), 1e-6)
        if ratio > MAPCFG['wide_region_ratio']:
            suspect[k] = f'region {ratio:.1f}x wider than its neighbours: a fret line was probably missed'
        elif ratio < MAPCFG['narrow_region_ratio']:
            suspect[k] = f'region only {ratio:.2f}x as wide as its neighbours: a spurious fret line is likely'
    model['region_suspect'] = suspect
    return model


def locate_string(model, P):
    # Which string is P closest to, and how clearly? Uses the real (possibly fanned-out) string lines.
    S, c_dir = model['strings'], model['c_dir']
    sd = [signed_dist(s, P, c_dir) for s in S]                       # positive = on the string-6 side of that string
    crossing = any(sd[i] <= sd[i + 1] for i in range(5))              # strings must stay in order at this point
    m = sum(d > 0 for d in sd)                                        # number of strings P is past
    if m == 0:                                                        # above string 1
        d1, gap = -sd[0], sd[0] - sd[1]
        near, d2 = 1, d1 + max(gap, 1e-6)
        outside = d1 > MAPCFG['outside_frac'] * max(gap, 1e-6)
        pos = 1 - d1 / max(gap, 1e-6)
    elif m == 6:                                                      # below string 6
        d1, gap = sd[5], sd[4] - sd[5]
        near, d2 = 6, d1 + max(gap, 1e-6)
        outside = d1 > MAPCFG['outside_frac'] * max(gap, 1e-6)
        pos = 6 + d1 / max(gap, 1e-6)
    else:                                                             # between string m and string m+1
        da, db = sd[m - 1], -sd[m]
        near = m if da <= db else m + 1
        d1, d2, gap = min(da, db), max(da, db), da + db
        outside = False
        pos = m + da / max(da + db, 1e-6)
    margin = (d2 - d1) / max(d2 + d1, 1e-6)
    return dict(index=int(near), position=float(pos), margin=float(margin), dist_px=float(d1), outside=bool(outside),
                crossing=bool(crossing), gap_px=float(gap))


def locate_fret(model, P):
    # Which fret region is P in (between which two boundaries), and how far from the nearer boundary?
    B, b_dir = model['boundaries'], model['b_dir']
    sd = [signed_dist(b, P, b_dir) for b in B]                       # positive = on the body side of that line
    k = sum(d > 0 for d in sd)                                        # number of boundaries P is past
    if k == 0:
        return dict(region='before_first_line', slot=None, margin=0.0, virtual_edge=False)
    if k == len(B):
        return dict(region='after_last_line', slot=None, margin=0.0, virtual_edge=False)
    dl, dr = sd[k - 1], -sd[k]                                        # distances to the boundary behind and ahead of P
    width = max(dl + dr, 1e-6)
    return dict(region='inside', slot=int(k), margin=float(min(dl, dr) / width), frac=float(dl / width),
                virtual_edge=bool(B[k - 1]['virtual'] or B[k]['virtual']), suspect=model['region_suspect'].get(int(k)))


def snap_to_string(model, P, index):
    # Foot of the perpendicular from P onto string `index`: the "mapped position" we draw.
    s = model['strings'][index - 1]
    d = _unit(s['p2'] - s['p1'])
    return (s['p1'] + ((P - s['p1']) @ d) * d)

def select_fretting_hand(hands, model):
    # Pick the hand most likely to be the fretting hand. Returns (index or None, number of other hands).
    if not hands['hands']:
        return None, 0
    best, best_key = 0, None
    for i, hd in enumerate(hands['hands']):
        near = 0
        if model is not None:
            poly = model['polygon'].astype(np.float32)
            for name in ('Index', 'Middle', 'Ring', 'Pinky'):
                t = hd['tips'][name]
                # signed distance to the polygon: > 0 inside, < 0 outside (in pixels)
                dist = cv2.pointPolygonTest(poly, (float(t['x_px']), float(t['y_px'])), True)
                near += dist >= -MAPCFG['near_neck_frac'] * model['spacing']
        key = (near, hd['score'])
        if best_key is None or key > best_key:
            best, best_key = i, key
    return best, len(hands['hands']) - 1


def map_fingertip(model, p2, finger, tip, hand_score):
    P = np.array([tip['x_px'], tip['y_px']], float)
    r = dict(finger=finger, landmark_id=tip['id'], x_px=tip['x_px'], y_px=tip['y_px'], proximity_only=True,
             status='no_geometry', decision='rejected', reason='', string_index=None, string_position=None,
             string_margin=None, fret_slot=None, fret_number=None, fret_margin=None, fret_frac=None, neck_coord=None,
             confidence=0.0, snap_xy=None)
    if model is None:
        r['reason'] = f"Phase 2 found no usable strings ({p2.get('failure_reason')})"
        return r
    s = locate_string(model, P)
    r.update(string_index=s['index'], string_position=round(s['position'], 2), string_margin=round(s['margin'], 2))
    if s['crossing']:
        r.update(status='ambiguous_string', reason='detected strings cross at this point; geometry unreliable here')
        return r
    if s['outside']:
        r.update(status='outside_fretboard', reason='beyond the outer string', string_index=None)
        return r
    f = locate_fret(model, P) if model['boundaries'] else None
    if f is not None and f['region'] == 'before_first_line':
        if model['nut_conf'] == 'medium':                              # a believable nut: beyond it is the headstock
            r.update(status='outside_fretboard', reason='behind the nut (headstock side)')
        else:                                                          # nut not confirmed: more fretboard may lie beyond
            r.update(status='beyond_detected_frets', reason='before the first detected line and the nut is not confirmed; region unknown')
        return r
    if f is not None and f['region'] == 'after_last_line':
        r.update(status='beyond_detected_frets', reason='past the last detected fret line; region width unknown')
        return r
    s_ok = s['margin'] >= MAPCFG['string_margin_min']
    if f is None:
        status = 'string_only' if s_ok else 'ambiguous_string'
        f_clear = 1.0
    else:
        f_ok = f['margin'] >= MAPCFG['fret_margin_min'] and not f['suspect']
        status = 'mapped' if (s_ok and f_ok) else 'ambiguous_both' if (not s_ok and not f_ok) else 'ambiguous_string' if not s_ok else 'ambiguous_fret'
        f_clear = float(np.clip(2 * f['margin'], 0, 1)) * (0.5 if f['virtual_edge'] else 1.0)
        r.update(fret_slot=f['slot'], fret_margin=round(f['margin'], 2),
                 fret_number=f['slot'] if model['nut_conf'] == 'medium' else None,
                 fret_frac=round(f['frac'], 3),                       # 0 = on the headstock-side line, 1 = on the body-side line
                 neck_coord=round(f['slot'] - 1 + f['frac'], 3))      # continuous position along the neck, in fret-region units
    gq = MAPCFG['geometry_quality'].get(p2.get('confidence'), 0.3)
    conf = gq * hand_score * float(np.sqrt(np.clip(s['margin'], 0, 1) * f_clear))
    r.update(status=status, confidence=round(float(conf), 3), snap_xy=[round(float(v), 1) for v in snap_to_string(model, P, s['index'])])
    if status.startswith('ambiguous'):
        r['reason'] = 'too close to the boundary between two ' + ('strings' if status == 'ambiguous_string' else 'fret regions' if status == 'ambiguous_fret' else 'strings and fret lines')
        if f is not None and f['suspect'] and status in ('ambiguous_fret', 'ambiguous_both'):
            r['reason'] = f"irregular fret spacing here ({f['suspect']})"
    elif finger == 'Thumb':
        r['reason'] = 'thumb usually rests behind the neck'
    elif p2.get('confidence') != 'high' and not MAPCFG['allow_low_geometry']:
        r.update(decision='tentative', reason='Phase 2 geometry is low confidence')
    elif conf < MAPCFG['accept_conf']:
        r.update(decision='tentative', reason=f"confidence {conf:.2f} below {MAPCFG['accept_conf']}")
    else:
        r.update(decision='accepted', reason='')
    return r


def map_image(image_id, bgr, hands, p2, p2_source='recompute', coord_problems=None):
    # Everything this needs is passed in (no notebook globals), so other notebooks can call it per image.
    model = build_neck_model(p2)
    h_idx, n_other = select_fretting_hand(hands, model)
    bbox = None
    if h_idx is not None:                                             # box around all 21 landmarks of the chosen hand
        lm = np.array(hands['hands'][h_idx]['landmarks_px'])
        bbox = [round(float(v), 1) for v in (lm[:, 0].min(), lm[:, 1].min(), lm[:, 0].max(), lm[:, 1].max())]
    out = dict(schema_version=2, image=dict(id=image_id, width=bgr.shape[1], height=bgr.shape[0]),
               phase1=dict(n_hands=len(hands['hands']), fretting_hand_index=h_idx, n_other_hands=n_other, hand_bbox=bbox),
               phase2=dict(source=p2_source, status=p2['status'], confidence=p2['confidence'], failure_reason=p2.get('failure_reason'),
                           neck_angle_deg=p2.get('neck_angle_deg')),
               geometry=dict(usable=model is not None, n_fret_boundaries=0 if not model else len(model['boundaries']),
                             n_virtual_boundaries=0 if not model else model['n_virtual'],
                             orientation='none' if not model else model['orientation'],
                             nut_confidence=None if not model else model['nut_conf'],
                             string_index_convention='1..6 in Phase 2 order (one side of the neck to the other); NOT guitar tuning',
                             fret_slot_convention='1 = region after the first detected headstock-side line; fret_number only when a medium-confidence nut anchors it'),
               coordinate_problems=coord_problems or [], fingertips=[], category='')
    if h_idx is None:
        out['category'] = 'no_hand'
        return out, model
    hd = hands['hands'][h_idx]
    for name in FINGERTIPS.values():
        r = map_fingertip(model, p2, name, hd['tips'][name], hd['score'])
        r['hand_score'] = hd['score']
        out['fingertips'].append(r)
    dec = [r['decision'] for r in out['fingertips']]
    stat = [r['status'] for r in out['fingertips']]
    if model is None:
        out['category'] = 'geometry_failed'
    elif 'accepted' in dec:
        out['category'] = 'success'
    elif 'tentative' in dec:
        out['category'] = 'tentative_low_geometry'
    elif any(s.startswith('ambiguous') for s in stat):
        out['category'] = 'ambiguous'
    else:
        out['category'] = 'outside_or_rejected'
    return out, model

## 3. Extraction (resumable and parallel)
**What:** `process_item` handles one image end to end and writes `outputs/phase4/features/<id>.json` containing the Phase 3 result, the full Phase 2 geometry, and the chord label.
**Technique: multiprocessing.** `multiprocessing.Pool` starts 2 worker processes; each one loads its **own** MediaPipe model (these objects must not be shared between processes) and then takes images from a queue. `fork` is used so the workers inherit all the functions defined above.
**Resumable:** a file is written only after the whole image is processed (written to a temporary name, then renamed), so a crash or disconnect never leaves a half-written file, and re-running skips finished images.
**Live progress:** a status line every few images: images done, speed, time left, and how the results are splitting into categories (`success`, `no_hand`, ...).

In [ ]:
import multiprocessing as mproc
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

LANDMARKER = None


def make_landmarker():
    # Exactly the Phase 1 settings: image mode, up to 2 hands, all confidences 0.3.
    opts = vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=vision.RunningMode.IMAGE, num_hands=2,
        min_hand_detection_confidence=0.3, min_hand_presence_confidence=0.3, min_tracking_confidence=0.3)
    return vision.HandLandmarker.create_from_options(opts)


def _init_worker():
    global LANDMARKER
    LANDMARKER = make_landmarker()            # one MediaPipe model per process


def load_bgr(item):
    # The ORIGINAL image as a BGR array. Nothing is resized here, so all coordinates stay in original pixels.
    if DATA_SOURCE == 'hf':
        pil = DS[item['split']][item['row']]['image'].convert('RGB')
        return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    with zipfile.ZipFile(ZIP_PATH) as z:
        return cv2.imdecode(np.frombuffer(z.read(item['zip_name']), np.uint8), cv2.IMREAD_COLOR)


def process_item(item):
    out = FEAT_DIR / f"{item['id']}.json"
    if out.exists():
        return item['id'], 'cached', 0.0
    t0 = time.time()
    bgr = load_bgr(item)
    hands = detect_hands(bgr, item['id'])                                       # Phase 1
    p2 = detect_fretboard(bgr, item['id'])                                      # Phase 2
    problems = check_coordinates(item['id'], bgr, hands, p2)                    # same pixel grid everywhere?
    res, _ = map_image(item['id'], bgr, hands, p2, 'recompute', problems)       # Phase 3
    res.update(chord=item['chord'], split=item['split'], row=item['row'], phase2_geometry=p2,
               n_hands_raw=[dict(score=h['score'], handedness=h['handedness_as_model_sees_it']) for h in hands['hands']])
    tmp = out.with_suffix('.tmp')
    tmp.write_text(json.dumps(res), encoding='utf-8')
    tmp.replace(out)                                                            # atomic: no half-written files
    return item['id'], res['category'], time.time() - t0


def run_extraction(items, n_workers=N_WORKERS, budget_min=TIME_BUDGET_MIN, every=10):
    todo = [it for it in items if not (FEAT_DIR / f"{it['id']}.json").exists()]
    print(f'{len(items) - len(todo)} already done, {len(todo)} to do, {n_workers} worker(s)')
    if not todo:
        return
    t_start, counts, n = time.time(), {}, 0
    if n_workers > 1:
        pool = mproc.get_context('fork').Pool(n_workers, initializer=_init_worker)
        stream = pool.imap_unordered(process_item, todo, chunksize=2)
    else:
        _init_worker(); pool = None
        stream = map(process_item, todo)
    try:
        for iid, cat, dt in stream:
            n += 1; counts[cat] = counts.get(cat, 0) + 1
            if n % every == 0 or n == len(todo):
                el = time.time() - t_start
                print(f'[{n}/{len(todo)}] {el / 60:.1f} min elapsed, {el / n:.2f} s/image, ~{(len(todo) - n) * el / n / 60:.1f} min left | {counts}', flush=True)
            if budget_min and (time.time() - t_start) / 60 > budget_min:
                print(f'time budget of {budget_min} min reached: re-run this cell to continue'); break
    finally:
        if pool is not None:
            pool.terminate(); pool.join()
    print('finished this run:', counts)


run_extraction(ITEMS)
print('JSON files now on disk:', len(list(FEAT_DIR.glob('*.json'))))

## 4. Build the feature table
**What:** read every JSON and flatten it into one row per image (`features.csv`).

**Columns, in plain words:**
- Identity: `id`, `chord` (the label), `split` (the dataset's own split), `row`.
- Availability flags: `hand_found`, `geom_usable` (6 strings found), `geom_conf` (`high`/`low`/`none` from Phase 2) and the number `geom_conf_num` (1.0 / 0.6 / 0), `n_accepted` fingertips.
- Per finger `F` in Index, Middle, Ring, Pinky, Thumb: `F_sp` (string position 1-6, continuous), `F_nc` (position along the neck in fret-region units), `F_conf` (Phase 3 confidence), `F_dec` (2 = accepted, 1 = tentative, 0 = rejected), `F_in` (1 if the tip is on the neck area).
- **Relative features:** differences between every pair of the four fretting fingers (`d_sp_Index_Middle`, `d_nc_...`), and each finger minus the lowest finger on the neck (`F_sp_rel`, `F_nc_rel`). A chord is a *shape*, so these should matter more than absolute positions.
- Crop box for the image models: `crop_x0..y1` plus `crop_kind` (`neck+hand`, `neck`, or `full` when nothing was found). Box = fretboard polygon plus hand box, padded 12% and clipped to the image.

**Missing values:** when a finger is not on the neck, or no hand/neck was found, the cells are left **empty (NaN)**. Nothing is invented. Gradient-boosted trees (notebook 04b) handle NaN natively.

In [ ]:
FINGERS = ['Index', 'Middle', 'Ring', 'Pinky', 'Thumb']
FRET4 = ['Index', 'Middle', 'Ring', 'Pinky']
DEC = {'accepted': 2, 'tentative': 1, 'rejected': 0}
IN_NECK = {'mapped', 'string_only', 'ambiguous_string', 'ambiguous_fret', 'ambiguous_both'}
nan = float('nan')


def crop_box(res):
    # Box (in original pixels) around the fretboard and the hand, padded and clipped to the image.
    W, H = res['image']['width'], res['image']['height']
    pts = []
    poly = (res.get('phase2_geometry') or {}).get('fretboard')
    kinds = []
    if poly:
        pts += [tuple(p) for p in poly['polygon']]; kinds.append('neck')
    bb = res['phase1'].get('hand_bbox')
    if bb:
        pts += [(bb[0], bb[1]), (bb[2], bb[3])]; kinds.append('hand')
    if not pts:
        return 0, 0, W, H, 'full'
    a = np.array(pts, float)
    x0, y0, x1, y1 = a[:, 0].min(), a[:, 1].min(), a[:, 0].max(), a[:, 1].max()
    px, py = 0.12 * (x1 - x0), 0.12 * (y1 - y0)
    box = [int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))]
    if box[2] - box[0] < 32 or box[3] - box[1] < 32:
        return 0, 0, W, H, 'full'
    return (*box, '+'.join(kinds))


def feature_row(res):
    p2, g = res['phase2'], res['geometry']
    tips = {t['finger']: t for t in res['fingertips']}
    row = dict(id=res['image']['id'], chord=res['chord'], split=res['split'], row=res['row'],
               width=res['image']['width'], height=res['image']['height'],
               hand_found=int(res['phase1']['fretting_hand_index'] is not None), n_hands=res['phase1']['n_hands'],
               geom_usable=int(g['usable']), geom_status=p2['status'], geom_conf=p2['confidence'],
               geom_conf_num={'high': 1.0, 'low': 0.6}.get(p2['confidence'], 0.0), category=res['category'],
               n_accepted=sum(t['decision'] == 'accepted' for t in res['fingertips']),
               nut_conf=g.get('nut_confidence') or 'none', n_boundaries=g['n_fret_boundaries'], n_virtual=g['n_virtual_boundaries'],
               hand_score=tips['Index']['hand_score'] if tips else nan)
    for f in FINGERS:
        t = tips.get(f)
        on_neck = t is not None and t['status'] in IN_NECK
        row[f'{f}_in'] = int(on_neck)
        row[f'{f}_sp'] = t['string_position'] if (t and (on_neck or t['status'] == 'beyond_detected_frets')) else nan
        row[f'{f}_nc'] = t['neck_coord'] if (on_neck and t['neck_coord'] is not None) else nan
        row[f'{f}_conf'] = t['confidence'] if t else nan
        row[f'{f}_dec'] = DEC[t['decision']] if t else nan
    for a_i, a in enumerate(FRET4):                      # differences between the fretting fingers: chords are shapes
        for b in FRET4[a_i + 1:]:
            row[f'd_sp_{a}_{b}'] = row[f'{a}_sp'] - row[f'{b}_sp']
            row[f'd_nc_{a}_{b}'] = row[f'{a}_nc'] - row[f'{b}_nc']
    sp = [row[f'{f}_sp'] for f in FRET4]; nc = [row[f'{f}_nc'] for f in FRET4]
    sp_min = np.nanmin(sp) if not np.all(np.isnan(sp)) else nan
    nc_min = np.nanmin(nc) if not np.all(np.isnan(nc)) else nan
    for f in FRET4:
        row[f'{f}_sp_rel'] = row[f'{f}_sp'] - sp_min
        row[f'{f}_nc_rel'] = row[f'{f}_nc'] - nc_min
    row['n_in_neck'] = sum(row[f'{f}_in'] for f in FRET4)
    row['features_usable'] = int(row['hand_found'] and row['n_in_neck'] >= 2)
    x0, y0, x1, y1, kind = crop_box(res)
    row.update(crop_x0=x0, crop_y0=y0, crop_x1=x1, crop_y1=y1, crop_kind=kind)
    return row


rows = []
for it in ITEMS:
    f = FEAT_DIR / f"{it['id']}.json"
    if f.exists():
        rows.append(feature_row(json.loads(f.read_text(encoding='utf-8'))))
FEATS = pd.DataFrame(rows)
FEATS.to_csv(P4 / 'features.csv', index=False)
print(FEATS.shape, '->', P4 / 'features.csv')
FEATS[['id', 'chord', 'hand_found', 'geom_conf', 'n_accepted', 'Index_sp', 'Index_nc', 'features_usable', 'crop_kind']].head(8)

## 5. Report: how much of the dataset do we actually get features for?
This is the honest picture that decides how much the geometry features can help in 04b. Read it before trusting the next notebook's results.
- **hand found:** MediaPipe returned a hand.
- **geometry high:** Phase 2 reported `high` confidence.
- **>=1 accepted:** at least one fingertip was `accepted` by Phase 3.
- **features usable:** a hand was found and at least 2 of the 4 fretting fingers are on the neck area, so the relative features can be computed.

In [ ]:
import matplotlib.pyplot as plt

def pct(s): return round(100 * float(np.mean(s)), 1)

by_chord = FEATS.groupby('chord').apply(lambda d: pd.Series({
    'images': len(d), 'hand found %': pct(d['hand_found']), 'geometry high %': pct(d['geom_conf'] == 'high'),
    '>=1 accepted %': pct(d['n_accepted'] > 0), 'features usable %': pct(d['features_usable'])}), include_groups=False)
total = pd.Series({'images': len(FEATS), 'hand found %': pct(FEATS['hand_found']), 'geometry high %': pct(FEATS['geom_conf'] == 'high'),
                   '>=1 accepted %': pct(FEATS['n_accepted'] > 0), 'features usable %': pct(FEATS['features_usable'])}, name='ALL')
REPORT = pd.concat([by_chord, total.to_frame().T])
print(REPORT.to_string())
print('\ncategories:', FEATS['category'].value_counts().to_dict())
print('crop kinds:', FEATS['crop_kind'].value_counts().to_dict())
print('coordinate problems (should be 0):', int(sum(bool(json.loads((FEAT_DIR / f"{i}.json").read_text())['coordinate_problems']) for i in FEATS['id'])))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
by_chord[['hand found %', 'geometry high %', 'features usable %']].plot.bar(ax=ax[0]); ax[0].set_title('availability by chord'); ax[0].set_ylabel('% of images')
FEATS['category'].value_counts().plot.barh(ax=ax[1], color='gray'); ax[1].set_title('Phase 3 image categories')
plt.tight_layout(); plt.show()

In [ ]:
# A look at the extraction: six images with the Phase 1-3 result drawn on the ORIGINAL photo
def quick_overlay(item):
    res = json.loads((FEAT_DIR / f"{item['id']}.json").read_text(encoding='utf-8'))
    bgr = load_bgr(item)
    out = draw_result(bgr, res['phase2_geometry'], alpha=0.18)            # Phase 2 drawing (region, strings, frets, nut)
    s = max(out.shape[:2]) / 640.0
    for t in res['fingertips']:
        c = {'accepted': (0, 200, 0), 'tentative': (0, 255, 255)}.get(t['decision'], (160, 160, 160))
        p = (int(round(t['x_px'])), int(round(t['y_px'])))
        cv2.circle(out, p, max(3, int(6 * s)), c, -1, cv2.LINE_AA); cv2.circle(out, p, max(3, int(6 * s)) + 1, (0, 0, 0), 1, cv2.LINE_AA)
    x0, y0, x1, y1, _ = crop_box(res)
    return out[y0:y1, x0:x1], f"{item['id']} | {res['category']}"

pick = [it for it in ITEMS if (FEAT_DIR / f"{it['id']}.json").exists()]
cats = {it['id']: json.loads((FEAT_DIR / f"{it['id']}.json").read_text(encoding='utf-8'))['category'] for it in pick}
show = []
for c in ('success', 'tentative_low_geometry', 'ambiguous', 'outside_or_rejected', 'geometry_failed', 'no_hand'):
    show += [it for it in pick if cats[it['id']] == c][:1]
show = (show + pick)[:6]
ims, ts = zip(*[quick_overlay(it) for it in show])
show_images(list(ims), list(ts), cols=3, size=4.6)

## 6. Save and download
**What:** packs `features.csv` and all per-image JSON files into `p4_features.zip` with a small `manifest.json` (settings, counts, library versions). Notebook 04b reads this zip: if you run 04b **in the same Colab session** it finds the files on disk and you do not need to upload anything; otherwise upload this zip there.
**Reminder:** Colab's disk is erased when the session ends, so download the zip before closing, especially after the long full run.

In [ ]:
manifest = dict(n_images=len(FEATS), smoke=SMOKE, data_source=DATA_SOURCE, schema_version=2, n_workers=N_WORKERS,
                counts=FEATS['category'].value_counts().to_dict(), mediapipe=mp.__version__,
                columns=list(FEATS.columns), created=time.strftime('%Y-%m-%d %H:%M:%S'))
(P4 / 'manifest.json').write_text(json.dumps(manifest, indent=1), encoding='utf-8')
zip_out = shutil.make_archive(str(Path(PROJECT_DIR) / 'p4_features'), 'zip', P4)
print('Created', zip_out, f'({os.path.getsize(zip_out) / 1e6:.1f} MB)')

In [ ]:
# colab-only
from google.colab import files
files.download(f'{PROJECT_DIR}/p4_features.zip')